In [63]:
import torch
from ngo.numericalanalysis.linalg import kron_matvec
import opt_einsum

In [105]:
N_samples = 10
n = (2,3,4)

# A_0 = torch.randn(N_samples, n[0], n[0], device='cuda', dtype=torch.float64)
# A_1 = torch.randn(N_samples, n[1], n[1], device='cuda', dtype=torch.float64)
# A_2 = torch.randn(N_samples, n[2], n[2], device='cuda', dtype=torch.float64)  

A_0 = torch.eye(n[0], device='cuda', dtype=torch.float64)[None,:]
A_1 = torch.eye(n[1], device='cuda', dtype=torch.float64)[None,:]
A_2 = torch.eye(n[2], device='cuda', dtype=torch.float64)[None,:]    

A_i = (A_0, A_1, A_2)

# x = torch.randn((N_samples, *n), device='cuda', dtype=torch.float64)
# x = torch.zeros((N_samples, n[0]*n[1]*n[2]), device='cuda', dtype=torch.float64)
# x[:,0] = 1
# x = x.reshape(N_samples, *n)
x = torch.ones((N_samples, *n), device='cuda', dtype=torch.float64)/torch.sqrt(torch.tensor(n[0]*n[1]*n[2], dtype=torch.float64))

In [106]:
x.shape

torch.Size([10, 2, 3, 4])

In [107]:
A_0

tensor([[[1., 0.],
         [0., 1.]]], device='cuda:0', dtype=torch.float64)

In [108]:
kron_matvec(A_i, x).shape

torch.Size([10, 2, 3, 4])

In [113]:
def kron_innerproduct(x, A_i, y):
    """Compute the inner product <x, (A_0 ⊗ A_1 ⊗ ... ⊗ A_{d-1}) y>
    for a batch of Kronecker-product matrices and vectors without forming
    the full Kronecker product.
    Args:
        x: Tensor of shape (N_samples, n_0, n_1, ..., n_{d-1})
        A_i: Tuple of d Tensors, each of shape (N_samples, n_i, n_i)
        y: Tensor of shape (N_samples, n_0, n_1, ..., n_{d-1})
    Returns:
        inner_products: Tensor of shape (N_samples,)
    """
    # Compute Ay = (A_0 ⊗ A_1 ⊗ ... ⊗ A_{d-1}) y
    Ay = kron_matvec(A_i, y)
    # Compute inner products <x, Ay>
    xAy = opt_einsum.contract('bi...,bi...->b', x, Ay)
    return xAy

In [114]:
kron_innerproduct_2(x, A_i, x) - kron_innerproduct(x, A_i, x)

torch.Size([10, 2, 3, 4])
torch.Size([10, 2, 3])
torch.Size([10, 2])
torch.Size([10])


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       dtype=torch.float64)